# UCF101 Action Recognition — Temporal Modeling Bake-off

**One-click walkthrough** of the [video-action-recognition-ai](https://github.com/Pranoy71/video-action-recognition-ai) repo:
three temporal modeling strategies on the same data, one evaluation protocol.

| model | temporal strategy | role |
|---|---|---|
| **A** ResNet-18 + mean pool | none (bag of frames) | the floor |
| **B** ResNet-18 + Transformer | attention over frames | learned temporal aggregation |
| **C** r3d_18 (Kinetics-400) | 3D convolutions, end-to-end | joint spatiotemporal features |

Plus: ONNX export + **INT8 static quantization** with measured accuracy/latency
trade-off on CPU.

**Runtime menu → Change runtime type → T4 GPU** (free tier is enough).

Expected wall-clock on a free T4: data ~10 min, A/B ~3 min, C ~20-40 min,
eval + quantize ~10 min.

In [ ]:
# 0) Setup — repo + dependencies
import os, sys

GIT_URL = "https://github.com/Pranoy71/video-action-recognition-ai"

if not os.path.exists("video-action-recognition-ai"):
    !git clone -q {GIT_URL}
%cd video-action-recognition-ai

!pip install -q av thop onnx onnxruntime onnxscript pyyaml tqdm

import torch, torchvision
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
!nvidia-smi -L 2>/dev/null || echo "CPU runtime (fine for A/B; C will be slow)"

## 1) Data — UCF101 subset, official splits

Primary source: the **official CRCV release** (UCF101.rar + the official
`ucfTrainTestlist` split files — the protocol from the dataset paper).
Fallback: the 10-class HF mirror (no RAR tooling needed).

The download script is idempotent and verifies per-split counts into
`data/ucf101/meta.json`. Validation is carved **by video group** out of the
official train list, so clips from the same source video never straddle
splits (leakage policy).

In [ ]:
# CRCV official (20 classes, ~6.9 GB download + extraction, ~10-15 min)
!python -m src.data.download_ucf101 --data-dir data --source crcv --num-classes 20

# If CRCV is slow / unrar is unavailable, fall back to the HF 10-class mirror:
# !python -m src.data.download_ucf101 --data-dir data --source hf

!cat data/ucf101/meta.json | python3 -c "import json,sys; d=json.load(sys.stdin); print('classes:', d['num_classes'], '| counts:', d['counts'])"

## 2) Cache frozen ResNet-18 features (models A/B)

One 16-frame pass per video through an ImageNet-pretrained ResNet-18.
The cache makes A-vs-B training iterate in **seconds** — and guarantees the
two models see byte-identical inputs, which is what makes their comparison
a controlled experiment rather than an anecdote.

In [ ]:
!python -m src.data.extract_features --data-dir data --out-dir data/features_resnet18 \
    --num-frames 16 --batch-size 32 --workers 2

## 3) Train A + B (fast)

Same optimizer family, same epoch budget, same label smoothing — the only
difference between A and B is **how frames are aggregated temporally**.

In [ ]:
!python -m src.train --config configs/framepool.yaml --data-dir data
!python -m src.train --config configs/temporal_transformer.yaml --data-dir data

## 4) Train C — r3d_18 fine-tuned end-to-end (the long part)

~20-40 min on a T4 with the default config (batch 16 + AMP + cosine
schedule, differential LRs: head 10x the pretrained backbone).

**Colab survival tip:** the trainer checkpoints full state
(`results/checkpoints/r3d18_state.pt`) after *every* epoch — if your session
drops, just re-run this cell with `--resume` and it continues where it left off.

In [ ]:
!python -m src.train --config configs/r3d18.yaml --data-dir data
# Re-run with resume after a session drop:
# !python -m src.train --config configs/r3d18.yaml --data-dir data --resume

## 5) Evaluate — the test split, touched exactly once per model

Top-1/Top-5, macro-F1, per-class recall, confusion matrix, plus the
efficiency bench: parameters, MACs/clip, model size, CPU throughput.
Frame-level models are benchmarked **as the full pipeline** (frozen
backbone + head), not just the cheap head on cached features.

In [ ]:
!python -m src.evaluate --config configs/framepool.yaml --data-dir data
!python -m src.evaluate --config configs/temporal_transformer.yaml --data-dir data
!python -m src.evaluate --config configs/r3d18.yaml --data-dir data

In [ ]:
# Inline the key plots
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("results/plots/*.png")):
    print(f)
    display(Image(f, width=700))

## 6) Quantization — ONNX + INT8 static, calibrated on TRAIN clips

Static (ahead-of-time) quantization converts the conv stack to int8 and
unlocks onnxruntime's quantized kernels — dynamic quantization would only
shrink the final Linear layer, a rounding error for a conv-dominated model.
Calibration uses train clips only (never test data: activation-scale
selection on test data is a subtle form of test leakage).

In [ ]:
!python -m src.export_quantize --config configs/r3d18.yaml --data-dir data
!cat results/logs/r3d18_quantization.json

## 7) Final report — README tables, generated from artifacts

In [ ]:
!python -m src.make_report
print(open("results/results_table.md").read())
print(open("results/quantization_table.md").read())

## 8) Single-video inference (the CLI a product would wrap)

In [ ]:
import glob
video = sorted(glob.glob("data/ucf101/videos/*/*.avi"))[0]
print("video:", video)
!python -m src.infer --video "{video}" --config configs/r3d18.yaml --data-dir data

## What to look at next

- `results/logs/*.json` — every metric, with the environment snapshot that produced it
- `README.md` — the write-up: architecture, evaluation, failure analysis, future work
- `src/` — every module has a "why" docstring; configs are the full experiment spec
- `tests/` — the synthetic-data suite that keeps the pipeline honest in CI